In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('ready_to_model_dataset.csv')

In [3]:
df.shape

(99946, 44)

In [4]:
df.columns

Index(['Age', 'Annual_Income', 'Num_Bank_Accounts', 'Num_Credit_Card',
       'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date',
       'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio',
       'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly',
       'Monthly_Balance', 'Credit_Mix_num', 'Payment_of_Min_Amount_num',
       'Credit_Score_num', 'Occupation_Architect', 'Occupation_Developer',
       'Occupation_Doctor', 'Occupation_Engineer', 'Occupation_Entrepreneur',
       'Occupation_Journalist', 'Occupation_Lawyer', 'Occupation_Manager',
       'Occupation_Mechanic', 'Occupation_Media_Manager',
       'Occupation_Musician', 'Occupation_Scientist', 'Occupation_Teacher',
       'Occupation_Writer', 'auto_loan', 'credit_builder_loan',
       'debt_consolidation_loan', 'home_equity_loan', 'mortgage_loan',
       'no_data', 'not_specified', 'payday_loan', 'personal_loan',
       'student

In [5]:
X = df.drop('Credit_Score_num',axis='columns')
X.shape

(99946, 43)

In [6]:
y = df.Credit_Score_num
y

0        2
1        2
2        2
3        2
4        2
        ..
99941    0
99942    0
99943    0
99944    1
99945    0
Name: Credit_Score_num, Length: 99946, dtype: int64

In [7]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, stratify=df.Credit_Score_num, test_size=0.2, random_state=21)

In [8]:
from xgboost import XGBClassifier
XGBClassifier()

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [9]:
xgb_model = XGBClassifier(n_estimators=500,
                          max_depth=8,
                          learning_rate=0.1,
                          objective='multi:softmax',
                          subsample=0.9,
                          colsample_bytree=0.7,
                          gamma=0,
                          num_class=3,
                          eval_metric='mlogloss',
                          verbosity=1,
                          tree_method='hist',
                          random_state=21,
                          )

In [10]:
xgb_model.fit(x_train,y_train)

,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [11]:
xgb_model.score(x_test,y_test)

0.8191095547773887

In [12]:
# now im gonna sample the data around 5% and try to find out roughly which model might be usefull

In [13]:
x_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.075, stratify=y, random_state=21)

In [14]:
print(x_sample.shape,y_sample.shape)

(7495, 43) (7495,)


In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = [
    {
        'name': 'RandomForest',
        'estimator': RandomForestClassifier(class_weight='balanced', random_state=42),
        'params': {
            'n_estimators': [50, 100, 250],
            'max_depth': [None, 10, 20]
        }
    },
    {
        'name': 'LogisticRegression',
        'estimator': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'params': {
            'C': [0.1, 1, 10],
            'solver': ['liblinear', 'lbfgs']
        }
    },
    {
        'name': 'SVC',
        'estimator': SVC(class_weight='balanced', probability=True, random_state=42),
        'params': {
            'kernel': ['linear', 'rbf'],
            'C': [1, 5, 10]
        }
    }
]



In [16]:
results = []

for m in models:
    print(f"Running GridSearchCV for {m['name']}...")
    gscv = GridSearchCV(
        estimator=m['estimator'],
        param_grid=m['params'],
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    gscv.fit(x_sample, y_sample)
    results.append({
        'model': m['name'],
        'best_score': gscv.best_score_,
        'best_params': gscv.best_params_
    })


Running GridSearchCV for RandomForest...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Running GridSearchCV for LogisticRegression...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Running GridSearchCV for SVC...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


In [17]:
import pandas as pd
results_df = pd.DataFrame(results).sort_values(by='best_score', ascending=False)
print(results_df)

                model  best_score                             best_params
0        RandomForest    0.719546  {'max_depth': 20, 'n_estimators': 250}
2                 SVC    0.683389               {'C': 5, 'kernel': 'rbf'}
1  LogisticRegression    0.657105             {'C': 1, 'solver': 'lbfgs'}


In [18]:
# lemme over sample and try it on the best model

In [19]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=21)
X_train_res, y_train_res = smote.fit_resample(x_train, y_train)


In [20]:
print(x_train.shape,y_train.shape)

(79956, 43) (79956,)


In [21]:
print(X_train_res.shape,y_train_res.shape)

(127560, 43) (127560,)


In [22]:
rf_model=RandomForestClassifier(max_depth=9,n_estimators=500,criterion='entropy',verbose=1)

In [23]:
RandomForestClassifier()

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
rf_model.fit(X_train_res,y_train_res)

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   21.3s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:  1.3min
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:  2.7min
[Parallel(n_jobs=1)]: Done 500 out of 500 | elapsed:  2.9min finished


,n_estimators,500
,criterion,'entropy'
,max_depth,9
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [25]:
rf_model.score(x_test,y_test)

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    1.2s
[Parallel(n_jobs=1)]: Done 500 out of 500 | elapsed:    1.4s finished


0.6861430715357679